> # **Exercícios - Aula 7** 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ry4n-felinto/impytech_2026/blob/main/aula_07/aula07_exercicios.ipynb)

 - **Professor:** Gustavo Cardoso 
    - al.gustavo.cardoso@impatech.edu.br

 - **Monitores:**
   - Bianca Zavadisk
      - al.bianca.abreu@impatech.edu.br
   - Bruno Pereira
      - al.bruno.paula@impatech.edu.br
   - Pedro Alberti
      - al.pedro.alberti@impatech.edu.br
   - Cristiane Sampaio
      - al.cristiane.sampaio@impatech.edu.br

- **Desenvolvedores:**
   - Bianca Zavadisk de Abreu
      - al.bianca.abreu@impatech.edu.br

 ## **Modelos e conceitos básicos de machine learning: modelos de classificação e regressão**

---

### ⚙️ **Preparação do Ambiente (Setup)**
Execute a célula abaixo para importar os pacotes necessários para a resolução dos problemas.

In [ ]:
!pip install xgboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessamento e Avaliação
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, r2_score, mean_squared_error

# Modelos e Transformadores
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, PolynomialFeatures
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

---

> ## **Problema 1 - Classificação, Análise Exploratória e Regularização** 🟡
**Revisão Teórica:** A **Regressão Logística** modela a probabilidade de um evento binário utilizando a função sigmoide. Quando trabalhamos com conjuntos de dados onde variáveis de entrada apresentam alta correlação entre si (multicolinearidade), o modelo tende a apresentar alta variância. Para mitigar este problema, aplicamos técnicas de **Regularização**: **L1 (Lasso)**, que adiciona uma penalidade absoluta podendo reduzir coeficientes a zero (seleção de características), e **L2 (Ridge)**, que adiciona uma penalidade quadrática, reduzindo-os proporcionalmente.

**Contextualização:** O conjunto de dados simulado a seguir representa 500 pacientes avaliados por meio de 15 exames laboratoriais ($X$). A variável dependente é binária (1 = Presença de doença, 0 = Ausência). Algumas variáveis são informativas, enquanto outras foram introduzidas como ruído correlacionado.

In [ ]:
from sklearn.datasets import make_classification

X_cls, y_cls = make_classification(n_samples=500, n_features=15, n_informative=5, n_redundant=8, random_state=42)

colunas_exames = [f'Exame_{i+1}' for i in range(15)]
df_pacientes = pd.DataFrame(X_cls, columns=colunas_exames)
df_pacientes['Diagnostico'] = y_cls
df_pacientes.head()

### 📝 **Problema 1.1 - Análise Exploratória: Distribuição e Multicolinearidade**
Antes do ajuste do modelo, é necessário compreender as características das variáveis e a estrutura de correlação dos dados.

**🎯 Objetivo:** Utilize a biblioteca `seaborn` (`sns.histplot` ou `sns.kdeplot`) para visualizar a distribuição da variável `Exame_1`, agrupada pela classe de diagnóstico. Em seguida, gere um mapa de calor (`sns.heatmap`) contendo a matriz de correlação das variáveis independentes (excluindo a coluna alvo).<br>

In [ ]:
# Área de código para o problema 1.1:
plt.figure(figsize=(16, 5))

# Gráfico 1: Distribuição
plt.subplot(1, 2, 1)
# [INSERIR CÓDIGO AQUI]
plt.title("Distribuição do Exame 1 por Diagnóstico")

# Gráfico 2: Correlação
plt.subplot(1, 2, 2)
matriz_corr = # [INSERIR CÓDIGO AQUI]
# [INSERIR CÓDIGO AQUI]
plt.title("Matriz de Correlação dos Exames")

plt.tight_layout()
plt.show()

**Pergunta:** Com base no mapa de calor gerado, identifique os sinais de multicolinearidade. Discuta os efeitos que a presença de variáveis altamente correlacionadas pode exercer sobre a estabilidade dos coeficientes em um modelo de regressão linear ou logística clássico.

**Resposta:** *(Clique duas vezes para editar)*

---

### 📝 **Problema 1.2 - Ajuste do Modelo de Regressão Logística Base**
Nesta etapa, estabeleceremos um modelo de referência para comparar posteriormente com os modelos regularizados.

**🎯 Objetivo:** Divida os dados em conjuntos de treinamento e teste (70/30). Ajuste um modelo `LogisticRegression` utilizando parâmetros padrão (ou uma regularização baixa, definindo um parâmetro $C$ elevado). Calcule e imprima as métricas no conjunto de teste: Acurácia, Precisão, Sensibilidade (Recall), F1-Score e Área sob a Curva ROC (AUC-ROC).<br>

In [ ]:
# Área de código para o problema 1.2:
X = df_pacientes.drop('Diagnostico', axis=1)
y = df_pacientes['Diagnostico']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y, test_size=0.3, random_state=42)

modelo_base = # [INSERIR CÓDIGO AQUI]
# [INSERIR CÓDIGO AQUI: Ajustar o modelo]

y_pred_base = # [INSERIR CÓDIGO AQUI]
y_prob_base = # [INSERIR CÓDIGO AQUI: predict_proba]

print(f"Acurácia: {# [INSERIR CÓDIGO AQUI]:.3f}")
print(f"Precisão: {# [INSERIR CÓDIGO AQUI]:.3f}")
print(f"Sensibilidade (Recall): {# [INSERIR CÓDIGO AQUI]:.3f}")
print(f"F1-Score: {# [INSERIR CÓDIGO AQUI]:.3f}")
print(f"AUC-ROC: {# [INSERIR CÓDIGO AQUI]:.3f}")

---

### 📝 **Problema 1.3 - Avaliação de Desempenho e Ajuste do Limiar de Decisão**
A classificação padrão em modelos logísticos ocorre por meio do limiar de probabilidade de $0.5$. Em aplicações médicas, o custo associado a falsos negativos e falsos positivos raramente é simétrico.

**🎯 Objetivo:** Calcule a Matriz de Confusão utilizando as previsões do modelo base. Em seguida, ajuste o limiar de decisão para $0.30$ (classificando como positivo se a probabilidade $\ge 0.30$). Gere a nova Matriz de Confusão e recalcule a métrica de Sensibilidade (Recall).<br>

In [ ]:
# Área de código para o problema 1.3:
matriz_original = # [INSERIR CÓDIGO AQUI]

y_pred_novo_limiar = # [INSERIR CÓDIGO AQUI]
matriz_nova = # [INSERIR CÓDIGO AQUI]
novo_recall = # [INSERIR CÓDIGO AQUI]

print("Matriz Original (Limiar 0.5):\n", matriz_original)
print("\nMatriz Ajustada (Limiar 0.3):\n", matriz_nova)
print(f"\nSensibilidade (Limiar 0.3): {novo_recall:.3f}")

**Pergunta:** Descreva o efeito numérico da alteração do limiar sobre as taxas de falsos positivos e falsos negativos. Discuta a adequação deste ajuste em um contexto de triagem para uma patologia de evolução clínica severa.

**Resposta:** *(Clique duas vezes para editar)*

---

### 📝 **Problema 1.4 - Aplicação de Regularização L1 e L2**
A aplicação de termos de penalidade tem como objetivo restringir o espaço paramétrico e induzir a estabilidade do modelo frente aos dados de entrada.

**🎯 Objetivo:** Ajuste um modelo logístico utilizando regularização L1 (`penalty='l1'`) e outro com regularização L2 (`penalty='l2'`). Utilize `solver='liblinear'` e o hiperparâmetro `C=0.05` para garantir uma restrição perceptível. Plote um gráfico de barras para comparar as magnitudes dos coeficientes resultantes de ambos os modelos.<br>

In [ ]:
# Área de código para o problema 1.4:
modelo_l1 = LogisticRegression(penalty='l1', solver='liblinear', C=0.05, random_state=42)
modelo_l2 = LogisticRegression(penalty='l2', solver='liblinear', C=0.05, random_state=42)

# [INSERIR CÓDIGO AQUI: Ajustar os modelos]

coef_l1 = modelo_l1.coef_[0]
coef_l2 = modelo_l2.coef_[0]

# [INSERIR CÓDIGO AQUI: Construir o gráfico de barras comparativo]
plt.figure(figsize=(10, 5))
plt.title("Comparação dos Coeficientes: Penalidade L1 vs L2")
plt.show()

---

### 📝 **Problema 1.5 - Comparativo de Generalização via AUC-ROC**
O efeito da regularização é observado na capacidade do modelo de performar bem em dados não vistos, evitando focar no ruído.

**🎯 Objetivo:** Calcule e reporte a métrica AUC-ROC no conjunto de teste para os três modelos desenvolvidos: Base, Regularizado L1 e Regularizado L2.<br>

In [ ]:
# Área de código para o problema 1.5:
auc_l1 = roc_auc_score(y_test_c, modelo_l1.predict_proba(X_test_c)[:, 1])
auc_l2 = roc_auc_score(y_test_c, modelo_l2.predict_proba(X_test_c)[:, 1])

print(f"AUC L1 (Lasso): {auc_l1:.3f}")
print(f"AUC L2 (Ridge): {auc_l2:.3f}")

**Pergunta:** Considerando a matriz de correlação (Problema 1.1) e o gráfico de coeficientes (Problema 1.4), justifique de que maneira o algoritmo com regularização L1 tratou as variáveis colineares e como essa característica afeta a interpretabilidade do modelo clínico.

**Resposta:** *(Clique duas vezes para editar)*

---

> ## **Problema 2 - Séries Temporais e o Algoritmo XGBoost** 🔴
**Revisão Teórica:** Modelos preditivos tradicionais assumem que as observações são independentes. Em séries temporais, existe autocorrelação: o estado no instante $t$ é dependente de $t-1, t-2$. Para contornar esta restrição, empregamos a engenharia de **Lags** (defasagens). O algoritmo **XGBoost** é fundamentado em árvores de decisão sequenciais. Dada a sua alta flexibilidade, o XGBoost está sujeito ao sobreajuste (*overfitting*), sobretudo em séries temporais, demandando o controle rigoroso de hiperparâmetros e processos estritos de separação cronológica.

**Contextualização:** Iremos monitorar uma variável clínica contínua fictícia, aferida diariamente.

In [ ]:
# Geração de dados simulados de monitoramento contínuo ao longo de 200 períodos
np.random.seed(42)
t = np.arange(200)
variavel_clinica = 15 + 10 * np.sin(0.1 * t) + 2 * np.random.randn(200)

df_ts = pd.DataFrame({'Valor_Atual': variavel_clinica})


### 📝 **Problema 2.1 e 2.2 - Engenharia de Características (Lags) e Divisão Temporal**
A modelagem preditiva neste cenário requer que as entradas do modelo representem o histórico imediato da variável em estudo.

**🎯 Objetivo:** Adicione ao `df_ts` as colunas `Lag_1` (período anterior) e `Lag_2` (dois períodos atrás). Remova os valores ausentes. Segmente o conjunto em treinamento (80%) e teste (20%), garantindo a **ordem cronológica** (sem embaralhamento) para prevenir *Data Leakage*.<br>

In [ ]:
# Área de código para o problema 2.1 e 2.2:
df_ts['Lag_1'] = # [INSERIR CÓDIGO AQUI]
df_ts['Lag_2'] = # [INSERIR CÓDIGO AQUI]
df_ts = # [INSERIR CÓDIGO AQUI: Tratamento de nulos]

X_ts = df_ts[['Lag_1', 'Lag_2']]
y_ts = df_ts['Valor_Atual']

X_train_ts, X_test_ts, y_train_ts, y_test_ts = # [INSERIR CÓDIGO AQUI: Divisão com manutenção cronológica]


---

### 📝 **Problema 2.3 e 2.4 - Ajuste do Modelo XGBoost e Controle de Hiperparâmetros**
Avaliá-se nesta fase a suscetibilidade do modelo ao sobreajuste caso restrições estruturais não sejam empregadas.

**🎯 Objetivo:** Ajuste um estimador `XGBRegressor()` padrão (sem limites). Ajuste um segundo estimador `XGBRegressor()` estabelecendo limites restritivos (exemplo: `max_depth=2`, `learning_rate=0.05`, `n_estimators=50`). Reporte as métricas $R^2$ para os conjuntos de treinamento e teste em ambos.<br>

In [ ]:
# Área de código para o problema 2.3 e 2.4:
modelo_xgb_livre = # [INSERIR CÓDIGO AQUI]
modelo_xgb_restrito = XGBRegressor(max_depth=2, learning_rate=0.05, n_estimators=50, random_state=42)

# [INSERIR CÓDIGO AQUI: Ajustar e prever para ambos os modelos]

print("XGBoost Base - Treino R²:", # [INSERIR], "| Teste R²:", # [INSERIR])
print("XGBoost Restrito - Treino R²:", # [INSERIR], "| Teste R²:", # [INSERIR])

---

### 📝 **Problema 2.5 - Análise de Importância das Variáveis**
Algoritmos baseados em árvores viabilizam a extração da importância relativa das variáveis com base na redução do critério de impureza em suas ramificações.

**🎯 Objetivo:** Imprima o atributo `.feature_importances_` gerado pelo modelo `XGBoost Restrito` para analisar a relevância de `Lag_1` em contraposição a `Lag_2`.<br>

In [ ]:
# Área de código para o problema 2.5:
importancias = # [INSERIR CÓDIGO AQUI]
print("Importâncias das Características (Lag_1, Lag_2):", importancias)

**Pergunta:** Argumente, sob a ótica de processos temporais e biológicos, o motivo estatístico subjacente à predominância estrutural do `Lag_1` sobre o `Lag_2` observada na métrica do modelo.

**Resposta:** *(Clique duas vezes para editar)*

---

> ## **Problema 3 - Regressão Não-Linear com Splines Cúbicas** 🔴
**Revisão Teórica:** A Regressão Linear Simples postula uma correlação retilínea. Em domínios clínicos, essa hipótese é usualmente inadequada. Na tentativa de capturar a não-linearidade, funções polinomiais globais (ex: grau $> 10$) introduzem variância significativa nos extremos (Fenômeno de Runge). A solução recai nas **Splines Cúbicas**, particionando o domínio da variável em intervalos demarcados por **Nós** (*Knots*). O modelo ajusta polinômios independentes dentro de cada intervalo, garantindo continuidade e suavidade estrutural.

**Contextualização:** Avaliaremos uma curva de Dose vs Reação de um composto fictício, demonstrando as vulnerabilidades dos ajustes puramente polinomiais em comparação com as Splines.

In [ ]:
# Geração de dados que simulam uma relação não linear e não monotônica (Dose vs Resposta)
np.random.seed(0)
X_dose = np.linspace(0, 10, 100).reshape(-1, 1)
y_reacao = np.sin(X_dose).ravel() * X_dose.ravel() + np.random.normal(0, 0.8, X_dose.shape[0])

# Definição do vetor expandido para visualização analítica (estendendo os limites originais)
X_dominio_teste = np.linspace(-1, 11, 500).reshape(-1, 1)


### 📝 **Problema 3.1 - Ajuste de Modelo Linear Simples**
Neste passo, o modelo base demonstrará o sob-ajuste (*underfitting*) inerente a aproximações paramétricas restritas.

**🎯 Objetivo:** Ajuste a função `LinearRegression()` aos dados experimentais (`X_dose`, `y_reacao`). Calcule os valores preditos para a faixa estendida presente no vetor `X_dominio_teste`.<br>

In [ ]:
# Área de código para o problema 3.1:
modelo_lin = LinearRegression()
# [INSERIR CÓDIGO AQUI: Processo de fit]
pred_lin = # [INSERIR CÓDIGO AQUI: Previsão no vetor X_dominio_teste]


---

### 📝 **Problema 3.2 - Ajuste de Modelo Polinomial e Fenômeno de Runge**
Avaliaremos o comportamento das estimativas nas extremidades ao elevar o grau da função global.

**🎯 Objetivo:** Construa um `pipeline` analítico conectando o pré-processador `PolynomialFeatures(degree=15)` ao método `LinearRegression()`. Proceda com o treinamento e compute as previsões para `X_dominio_teste`.<br>

In [ ]:
# Área de código para o problema 3.2:
modelo_poly = # [INSERIR CÓDIGO AQUI]
# [INSERIR CÓDIGO AQUI]
pred_poly = # [INSERIR CÓDIGO AQUI]


---

### 📝 **Problema 3.3 - Ajuste de Modelo com Splines Cúbicas**
Aplicaremos a partição do espaço preditivo com continuidade rigorosa.

**🎯 Objetivo:** Declare um pipeline contendo `SplineTransformer(n_knots=5, degree=3, extrapolation='linear')` e `LinearRegression()`. Realize o ajuste sobre os dados formativos e salve as estimativas derivadas sobre o eixo projetado.<br>

In [ ]:
# Área de código para o problema 3.3:
modelo_spline = # [INSERIR CÓDIGO AQUI]
# [INSERIR CÓDIGO AQUI]
pred_spline = # [INSERIR CÓDIGO AQUI]


---

### 📝 **Problema 3.4 - Visualização Comparativa dos Modelos**
A eficácia dos métodos frente aos dados limítrofes e extrapolativos será mensurada por via gráfica.

**🎯 Objetivo:** Produza o diagrama de dispersão relacionando a matriz experimental. Projete as séries geradas de `pred_lin`, `pred_poly` e `pred_spline` no mesmo sistema de coordenadas. Acrescente marcações limitando o espaço empírico (de 0 a 10) para evidenciar a extrapolação.<br>

In [ ]:
# Área de código para o problema 3.4:
plt.figure(figsize=(12, 6))

plt.scatter(X_dose, y_reacao, color='black', alpha=0.5, label='Observações Clínicas')

# [INSERIR CÓDIGO AQUI: Linha preditiva do Modelo Linear Simples]
# [INSERIR CÓDIGO AQUI: Linha preditiva do Modelo Polinomial (Grau 15)]
# [INSERIR CÓDIGO AQUI: Linha preditiva do Modelo de Spline Cúbica]

plt.axvline(x=0, color='gray', linestyle='--')
plt.axvline(x=10, color='gray', linestyle='--')
plt.ylim(-15, 15)
plt.title("Avaliação Comparativa de Ajustes Paramétricos e Não Paramétricos")
plt.legend()
plt.show()

---

### 📝 **Problema 3.5 - Interpretação do Comportamento Extrapolativo**

**🎯 Objetivo:** Responder discursivamente os questionamentos baseando-se no comportamento visualizado no Problema 3.4.<br>

**Pergunta:** 
1. Avalie as previsões do modelo polinomial nos subespaços restritos à extrapolação ($X > 10$ e $X < 0$). Correlacione suas observações ao conceito de variância paramétrica.
2. Fundamente o motivo matemático pelo qual a abordagem de Spline Cúbica, conforme especificada, conseguiu delimitar as oscilações periféricas mantendo o rigor ajustado ao centro da distribuição empírica.

**Resposta:** *(Clique duas vezes para editar)*

---

> ## 🔗 **Conteúdo extra**
**Caso você queira praticar mais:**
- [Documentação Scikit-Learn: XGBoost vs Gradient Boosting](https://xgboost.readthedocs.io/en/stable/)
- [Scikit-Learn Documentation: Spline Features](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.SplineTransformer.html)
- [Scikit-Learn Classification Metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)

**Referências e leituras extras:**
- [Entendendo o Fenômeno de Runge](https://pt.wikipedia.org/wiki/Fen%C3%B4meno_de_Runge)